<a href="https://colab.research.google.com/github/G0nkly/pytorch_sandbox/blob/main/gpts/nanoGPT/AK_GPT_DIY_RECAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load the data
# Encoding / Decoding
# Train/Test Split | get_batch function


# Remember:
# * what to watch out for in k,v,q
# * comment shapes for Attention
# * whats important at the end of MHA?
# * what needs to be on the device? (Model, Data, ...)
# * evaluate on train and test
# * also show the numbers (sum) of parameters (numel()) of the model

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt

--2026-11-25 16:58:36--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-11-25 16:58:36 (123 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
with open("./input.txt", "r") as file:
  text = file.read()

text[:10]

vocab = sorted(list(set(text)))

stoi = {v:k for k, v in enumerate(vocab)}
itos = {k:v for k, v in enumerate(vocab)}

encode = lambda seq: [stoi[char] for char in seq]
decode = lambda ids: "".join([itos[id] for id in ids])

data = torch.tensor(encode(text))
train_test_split = 0.8
test_start_index = int(len(data) * 0.8)
train = data[:test_start_index]
test = data[test_start_index:]

In [5]:
###################
# HYPERPARAMETERS #
###################
vocab_size = len(vocab)
batch_size = 8
block_size = 8
learning_rate = 1e-03
device = "cuda" if torch.cuda.is_available() else "cpu"
n_epochs = 1000
n_evals = 100
eval_iter = 200
n_embed = 32
n_heads = 4

In [6]:
def get_batch(split):
  data = train if split == "train" else test
  idx = torch.randint(0, len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i + block_size] for i in idx])
  y = torch.stack([data[i + 1:i + block_size + 1] for i in idx])
  x.to(device), y.to(device)
  return x, y

In [7]:
def train_model(model, epochs=n_epochs):
  model.train()
  optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate)
  for epoch in n_epochs:
    x_train, y_train = get_batch("train")
    logits = model(x_train)
    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    y_train = y_train.view(B*T)
    loss = F.cross_entropy(logits, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if epoch % eval_iter == 0 or epoch == n_epochs - 1:
      eval_result = evaluate_model(model)
      print(f"Train loss: {eval_result['train']}, Test loss: {eval_result['test']}")

In [8]:
def evaluate_model(model):
  model.eval()
  out = {}
  datasets = ["train", "test"]
  for dataset in datasets:
    losses = torch.zeros(n_evals)
    for i in n_evals:
      xb, yb = get_batch(dataset)
      logits = model(xb)
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      yb = yb.view(B*T)
      loss = F.cross_entropy(logits, yb)
      losses[i] = loss

    out[dataset] = loss

  model.train()
  return out

In [9]:
class BigramModel(nn.Module):

  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    return x

  def generate(self, start_token, n_generations):
    for _ in range(n_generations):
      input = start_token[:, -block_size:]                       # B, block_size, C
      logits = self(input)                                      # B, T, C
      logits = logits[:, -1, :]                                 # B, 1, C
      probs = F.softmax(logits, dim=-1)                         # B, 1, C
      next_idx = torch.multinomial(probs, num_samples=1)       # B, 1, 1
      start_token = torch.cat((start_token, next_idx), dim=-1)  # B, T+1, C

    return start_token

In [10]:
model = BigramModel()
start_token = torch.tensor([0]).unsqueeze(0)
num_generations = 100
tokens = model.generate(start_token=start_token, n_generations=num_generations)
tokens = tokens.squeeze(0)
decode(tokens.tolist())

'\nVT::SeBYCs.Eer\nBEjmYPWnU3-oe..tsty,r:UhYaSv$Hh-3:&ziYlYreMQ,.DG,KEBdZtURvL:kPdXk;QNan.xs!iXJeU&xPXJu'

In [16]:
class AttentionHead(nn.Module):

  def __init__(self):
    super().__init__()
    self.head_size = n_embed // n_heads
    self.query = nn.Linear(n_embed, self.head_size)
    self.key = nn.Linear(n_embed, self.head_size)
    self.value = nn.Linear(n_embed, self.head_size)
    self.register_buffer("tril", torch.tril(torch.ones((block_size, block_size), device=device)))

  def forward(self, x):
    B, T, C = x.shape
    query = self.query(x)
    key = self.key(x)
    value = self.value(x)
    weights = (query @ torch.transpose(key, -2, -1)) / (self.head_size)**0.5
    weights = weights.masked_fill(self.tril[:T,:T] == 0, float("-inf"))
    weights = F.softmax(weights, dim=-1)
    return weights @ value

In [24]:
class MultiHeadAttention(nn.Module):

  def __init__(self):
    super().__init__()
    self.attention_heads = nn.ModuleList([AttentionHead() for _ in range(n_heads)])
    self.projection = nn.Linear(n_embed, n_embed)

  def forward(self, x):
    x = torch.cat([head(x) for head in self.attention_heads], dim=-1)
    x = self.projection(x)
    return x

In [27]:
class AttentionBlock(nn.Module):

  def __init__(self):
    super().__init__()

  def forward(self, x):
    ...

In [22]:
class ZeTransformer(nn.Module):

  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embed)
    self.positional_encoding = nn.Embedding(block_size, n_embed)
    self.projection = nn.Linear(n_embed, vocab_size)
    self.mha = MultiHeadAttention()

  def forward(self, x):
    x = self.embedding(x)
    x = x + self.positional_encoding(torch.arange(0, block_size, device=device))
    x = self.mha(x)
    x = self.projection(x)
    return x


In [23]:
transformer = ZeTransformer().to(device)
output = transformer(start_token.to(device))
output.shape

torch.Size([1, 8, 65])

In [14]:
torch.tril(torch.ones((block_size, block_size), device=device))

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])